---
## 9. Preprocesado y Encoding

### 9.1 Estrategia de imputación de valores nulos

| Variable | Tipo nulo | Estrategia | Justificación |
|----------|-----------|------------|---------------|
| `ESTUDIOS_PADRE` / `ESTUDIOS_MADRE` | Nulos reales + código 9 (NS/NC) | Imputar con **moda** ó tratar NS/NC como categoría propia | Pérdida de info si eliminamos; la moda es robusta para ordinales con distribución clara |
| `MOV_IN` | Pocos nulos esperados | Imputar con **moda** | Variable binaria, baja tasa de nulos |
| `TIC` | Pocos nulos esperados | Imputar con **moda** | Autoevaluación ordinal |
| Resto vars. clave | Baja tasa | Imputar con **moda** | Consistencia del pipeline |

### 9.2 Estrategia de Encoding

| Variable | Tipo | Encoding elegido | Justificación |
|----------|------|-----------------|---------------|
| `SEXO` | Binaria nominal | **One-Hot Encoding** (drop_first=True) | Evita multicolinealidad; sólo 1 dummy necesaria |
| `RAMA` | Nominal (5 cat.) | **One-Hot Encoding** | Sin orden natural entre ramas |
| `T_UNIV` | Nominal (4 cat.) | **One-Hot Encoding** | Sin orden natural; posible simplificación a binaria pública/privada |
| `MOV_IN` | Binaria nominal | **Label Encoding** (1/0) | Binaria, directo |
| `JORNADA` | Binaria nominal | **Label Encoding** (1/0) | Binaria, directo |
| `TIC` | Ordinal (3 niveles) | **Ordinal Encoding** [1,2,3] | Respeta el orden Básico < Intermedio < Avanzado |
| `ESTUDIOS_PADRE` / `MADRE` | Ordinal (8 niveles) | **Ordinal Encoding** [1–8] | Orden formativo real; NS/NC (9) → imputar antes |
| `TR_SUELDO` (target) | Ordinal (7 niveles) | Mantener **numérico** [1–7] | Los modelos ordinales y de boosting lo procesan directamente |

In [ ]:
# ── 9.1 Selección de columnas de modelado ────────────────────────────────────
import os
os.makedirs('../outputs/figures', exist_ok=True)

FEATURES = [v for v in VARS_ORIGINALES if v in df_model.columns]
TARGET   = 'TR_SUELDO_NUM'

df_prep = df_model[FEATURES + [TARGET]].copy()

# Convertir todo a numérico
for col in FEATURES:
    df_prep[col] = pd.to_numeric(df_prep[col], errors='coerce')

print(f'Shape antes de imputación: {df_prep.shape}')
print('Nulos por columna:')
print(df_prep[FEATURES].isna().sum())

Shape antes de imputación: (9773, 9)
Nulos por columna:
SEXO               0
RAMA               0
T_UNIV             0
ESTUDIOS_PADRE     0
ESTUDIOS_MADRE     0
MOV_IN            24
TIC                0
JORNADA            0
dtype: int64


In [ ]:
# ── 9.2 Tratamiento NS/NC en ESTUDIOS_PADRE y ESTUDIOS_MADRE ─────────────────
for col in ['ESTUDIOS_PADRE', 'ESTUDIOS_MADRE']:
    if col in df_prep.columns:
        # El código 9 es NS/NC → lo tratamos como NaN para imputar
        df_prep[col] = df_prep[col].replace(9, np.nan)

print('Nulos tras reemplazar NS/NC (9) por NaN en estudios familiares:')
print(df_prep[['ESTUDIOS_PADRE', 'ESTUDIOS_MADRE']].isna().sum() 
      if 'ESTUDIOS_PADRE' in df_prep.columns else 'Variables no encontradas')

Nulos tras reemplazar NS/NC (9) por NaN en estudios familiares:
ESTUDIOS_PADRE    552
ESTUDIOS_MADRE    363
dtype: int64


In [ ]:
# ── 9.3 Imputación con la moda ────────────────────────────────────────────────
imputer = SimpleImputer(strategy='most_frequent')
df_prep[FEATURES] = imputer.fit_transform(df_prep[FEATURES])

print('Nulos tras imputación:')
print(df_prep[FEATURES].isna().sum())
print(f'\nShape tras imputación: {df_prep.shape}')

Nulos tras imputación:
SEXO              0
RAMA              0
T_UNIV            0
ESTUDIOS_PADRE    0
ESTUDIOS_MADRE    0
MOV_IN            0
TIC               0
JORNADA           0
dtype: int64

Shape tras imputación: (9773, 9)


In [ ]:
# ── 9.4 Encoding ────────────────────────────────────────────────────────────
# Variables ordinales: mantener valores numéricos ya correctos [1–N]
VARS_ORDINAL = [v for v in ['TIC', 'ESTUDIOS_PADRE', 'ESTUDIOS_MADRE'] if v in FEATURES]

# Variables nominales: One-Hot Encoding
VARS_OHE = [v for v in ['SEXO', 'RAMA', 'T_UNIV'] if v in FEATURES]

# Variables binarias: Label Encoding (ya son 1/2, recodificar a 0/1)
VARS_BIN = [v for v in ['MOV_IN', 'JORNADA'] if v in FEATURES]
for col in VARS_BIN:
    # 1 → 1 (sí/completa), 2 → 0 (no/parcial)
    df_prep[col] = (df_prep[col] == 1).astype(int)

# One-Hot Encoding
df_encoded = pd.get_dummies(df_prep, columns=VARS_OHE, drop_first=False, dtype=int)

print('Shape tras encoding:', df_encoded.shape)
print('\nNuevas columnas tras OHE:')
nuevas = [c for c in df_encoded.columns if any(c.startswith(v+'_') for v in VARS_OHE)]
print(nuevas)

Shape tras encoding: (9773, 17)

Nuevas columnas tras OHE:
['SEXO_1.0', 'SEXO_2.0', 'RAMA_1.0', 'RAMA_2.0', 'RAMA_3.0', 'RAMA_4.0', 'RAMA_5.0', 'T_UNIV_1.0', 'T_UNIV_2.0', 'T_UNIV_3.0', 'T_UNIV_4.0']


In [ ]:
# ── 9.5 Guardar dataset preprocesado ─────────────────────────────────────────
import os
os.makedirs('../outputs', exist_ok=True)

df_encoded.to_csv('../../Data/df_preprocesado.csv', index=False)
print(f'✅ Dataset preprocesado guardado en ../outputs/df_preprocesado.csv')
print(f'   Shape final: {df_encoded.shape}')
df_encoded.head(3)

✅ Dataset preprocesado guardado en ../outputs/df_preprocesado.csv
   Shape final: (9773, 17)


,ESTUDIOS_PADRE,ESTUDIOS_MADRE,MOV_IN,TIC,JORNADA,TR_SUELDO_NUM,SEXO_1.0,SEXO_2.0,RAMA_1.0,RAMA_2.0,RAMA_3.0,RAMA_4.0,RAMA_5.0,T_UNIV_1.0,T_UNIV_2.0,T_UNIV_3.0,T_UNIV_4.0
0,8.00,8.00,1,2.00,0,5,1,0,0,0,0,0,1,0,0,1,0
1,8.00,7.00,1,2.00,0,5,1,0,0,0,1,0,0,0,0,0,1
2,7.00,6.00,0,3.00,0,5,1,0,0,0,0,1,0,0,0,0,1


In [ ]:
# ── 9.6 Resumen de columnas del dataset de modelado ──────────────────────────
print('Columnas del dataset de modelado:')
for col in df_encoded.columns:
    dtype = df_encoded[col].dtype
    nuniq = df_encoded[col].nunique()
    print(f'  {col:<35} dtype={str(dtype):<8} nunique={nuniq}')

Columnas del dataset de modelado:
  ESTUDIOS_PADRE                      dtype=float64  nunique=8
  ESTUDIOS_MADRE                      dtype=float64  nunique=8
  MOV_IN                              dtype=int64    nunique=2
  TIC                                 dtype=float64  nunique=4
  JORNADA                             dtype=int64    nunique=2
  TR_SUELDO_NUM                       dtype=int64    nunique=7
  SEXO_1.0                            dtype=int64    nunique=2
  SEXO_2.0                            dtype=int64    nunique=2
  RAMA_1.0                            dtype=int64    nunique=2
  RAMA_2.0                            dtype=int64    nunique=2
  RAMA_3.0                            dtype=int64    nunique=2
  RAMA_4.0                            dtype=int64    nunique=2
  RAMA_5.0                            dtype=int64    nunique=2
  T_UNIV_1.0                          dtype=int64    nunique=2
  T_UNIV_2.0                          dtype=int64    nunique=2
  T_UNIV_3.0         